In [6]:
#!/usr/bin/env python3
"""plot_net_range_shared.py — RQ3 network figure, two panels with ONE shared legend.

    python3 plot_net_range_shared.py  CSV_DIR  OUT_DIR  [--rtts 1 10] [--metric bytes]

  In Jupyter: run the cells, then   main(["rq3_net_alg3_off100_out", "net_figs_shared"])

CSV_DIR holds net_range_rtt<R>_bw<B>.csv (one row per system x setting x seed).
Point = mean over seeds, error bar = +-1 sd.

Panels: (a) skewed data, matched queries   (b) skewed data, mismatched queries.
Lines : SidePart Alg. 2 and SidePart Oblivious Sharding (eps=0), one per RTT.

Writes to OUT_DIR, each as .pdf and .png:
  net_range_<metric>_shared         the 1x2 figure, legend once on top (use this one)
  net_range_<metric>_skew_matched   panel (a) alone, no legend  } for LaTeX subfigures
  net_range_<metric>_skew_mismatched panel (b) alone, no legend } + the legend file below
  net_range_<metric>_legend         the legend alone (put it above the two subfigures)
metric: bytes (network KB / admitted txn, default) | tput | shards
"""
import argparse, csv, glob, os, re, statistics, sys
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [7]:
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8, "legend.fontsize": 7,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "pdf.fonttype": 42, "ps.fonttype": 42})

PANELS = [  # csv setting, panel title
    ("skewed_data+matched_queries",    "Skewed data, matched queries"),
    ("skewed_data+mismatched_queries", "Skewed data, mismatched queries"),
]
# (system in csv, legend text, colour first RTT, colour later RTTs, marker, hollow marker?)
SYSTEMS = [
    ("DePart", "SidePart, Alg. 2 ($\\epsilon$={eps})",                 "#0b4f9c", "#5b8fd0", "o", False),
    ("Random", "SidePart, Oblivious Sharding ($\\epsilon$=0)", "#ff7f0e", "#ffad66", "s", True),
]
DP_STYLE   = ["-", "-.", ":", (0, (5, 2, 1, 2))]   # line style per RTT (first, second, ...)
ZERO_STYLE = ["--", ":", "-.", (0, (1, 1))]
RTT_MARKER = ["o", "^", "D", "v"]
METRICS = {   # metric -> (row key, y label, log y?)
    "bytes":  ("kb",     "Network KB / admitted txn"),
    "tput":   ("tput",   "Committed throughput (txn/s)"),
    "shards": ("shards", "Shards contacted per round"),
}

In [8]:
def load(csv_dir):
    data = defaultdict(list)       # (system, setting, rtt, bw) -> list of row dicts (one per seed)
    files = [f for f in glob.glob(os.path.join(csv_dir, "**", "net_range_rtt*_bw*.csv"), recursive=True)
             if not f.endswith("_subepoch.csv")]
    eps = {}
    for f in files:
        m = re.search(r"net_range_rtt(\d+(?:\.\d+)?)_bw(\d+(?:\.\d+)?)\.csv$", f)
        if not m:
            continue
        rtt, bw = float(m.group(1)), float(m.group(2))
        with open(f) as fh:
            for r in csv.DictReader(fh):
                txns = float(r["txns"]) or 1.0
                k = float(r["k"])
                data[(r["system"], r["setting"], rtt, bw)].append({
                    "tput": float(r["throughput_txn_s"]),
                    "kb": float(r["net_bytes"]) / txns / 1024.0,       # KB per admitted txn
                    "shards": float(r["contacted_frac"]) * k,          # shards contacted per round
                })
                eps.setdefault(r["system"], r["epsilon"])
    print(f"  {len(files)} csv files, {sum(len(v) for v in data.values())} rows")
    return data, eps


def ms(vals):
    return (statistics.fmean(vals), statistics.stdev(vals) if len(vals) > 1 else 0.0) if vals else (float("nan"), 0.0)

In [9]:
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("csv_dir")
    ap.add_argument("out")
    ap.add_argument("--rtts", type=float, nargs="+", default=[1, 10], help="RTTs (ms) to draw")
    ap.add_argument("--metric", choices=list(METRICS), default="bytes")
    ap.add_argument("--width", type=float, default=7.0, help="width of the 1x2 figure (in)")
    ap.add_argument("--height", type=float, default=2.5)
    a = ap.parse_args(argv)
    a.csv_dir, a.out = os.path.expanduser(a.csv_dir), os.path.expanduser(a.out)
    data, eps = load(a.csv_dir)
    if not data:
        print("!! no net_range_rtt*_bw*.csv in", os.path.abspath(a.csv_dir), " (cwd:", os.getcwd() + ")"); return
    os.makedirs(a.out, exist_ok=True)
    print("  output ->", os.path.abspath(a.out))
    key, ylabel = METRICS[a.metric]
    bws = sorted({k[3] for k in data})
    rtts = [r for r in a.rtts if r in {k[2] for k in data}]
    rt = ", ".join(f"{r:g}" for r in rtts)

    def panel(ax, setting, title):
        for si, (sysname, base, c0, c1, mk, hollow) in enumerate(SYSTEMS):
            for ri, rtt in enumerate(rtts):
                color = c0 if ri == 0 else c1
                xs, ys, es = [], [], []
                for bw in bws:
                    m, s = ms([r[key] for r in data.get((sysname, setting, rtt, bw), [])])
                    if m == m:
                        xs.append(bw); ys.append(m); es.append(s)
                ax.errorbar(xs, ys, yerr=es, color=color, linestyle=(DP_STYLE if si == 0 else ZERO_STYLE)[ri % 4],
                            marker=mk if ri == 0 else RTT_MARKER[ri % 4], markersize=6, capsize=2,
                            linewidth=1.6, alpha=1.0 if ri == 0 else 0.75,
                            markerfacecolor="white" if hollow else color,
                            label=base.format(eps=eps.get(sysname, "?")) + f", RTT {rtt:g} ms")
        ax.set_xscale("log")
        ax.set_ylim(bottom=0)
        ax.set_xlabel(f"Bandwidth (Mbit/s), RTT = {rt} ms")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.3, linewidth=0.6)

    def save(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(a.out, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        print("  wrote", name)

    base = f"net_range_{a.metric}"
    # 1) the 1x2 figure with one legend on top (same y axis, so the panels compare directly)
    fig, axes = plt.subplots(1, 2, figsize=(a.width, a.height), sharey=True)
    for ax, (setting, title) in zip(axes, PANELS):
        panel(ax, setting, title)
    axes[1].set_ylabel("")
    axes[1].tick_params(labelleft=True)
    h, l = axes[0].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=7.5)
    save(fig, f"{base}_shared")

    # 2) each panel alone without a legend + the legend as its own file (LaTeX subfigures)
    ymax = max(ax.get_ylim()[1] for ax in axes)
    for (setting, title), suffix in zip(PANELS, ("skew_matched", "skew_mismatched")):
        fig, ax = plt.subplots(figsize=(3.4, 2.45))
        panel(ax, setting, title)
        ax.set_ylim(0, ymax)
        fig.tight_layout()
        save(fig, f"{base}_{suffix}")
    fig = plt.figure(figsize=(a.width, 0.45))
    fig.legend(h, l, loc="center", ncol=2, frameon=False, fontsize=7.5)
    save(fig, f"{base}_legend")


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [ ]:
main(["rq3_net_alg3_off100_out", "net_figs_shared"])